In [1]:
import h5py
import torch
import numpy as np
import os
import json

import keyrank_rs

import torch.nn as nn

from tqdm import tqdm

In [2]:
if os.name == "nt":
    DATAFOLDER = "C:/Data"
else:
    DATAFOLDER = "/mnt/c/Data"

val_test_hdf = h5py.File(f"{DATAFOLDER}/simpleserial-aes-fix-500-diff.hdf5")

val_test_traces = torch.Tensor(np.array(val_test_hdf['trace']))
val_test_plaintexts = torch.Tensor(np.array(val_test_hdf['data']))
val_test_keys = torch.Tensor(np.array(val_test_hdf['key']))

device = torch.device("cuda")

In [3]:
print(val_test_traces.shape)
print(val_test_plaintexts.shape)
print(val_test_keys.shape)

torch.Size([1000, 500, 5000])
torch.Size([1000, 500, 32])
torch.Size([1000, 16])


In [4]:
def metadata_best_epoch(model_name) -> int:
    with open(f"models/{model_name}/metadata.json") as f:
        metadata = json.load(f)
        val_scores = metadata["scores"][1]
        best_epoch = np.array(val_scores).argmin()
    return best_epoch.item()

def get_traces_mean_std(trace_start, trace_end):
    """Load the mean and std of the training trace set within the given interval"""
    with open(f"misc/standardization/trace{trace_start}_{trace_end}.json") as f:
        info = json.load(f)
        mean = info['training_traces_mean']
        std = info['training_traces_std']

    return mean, std

In [11]:
IMPL = "fixslice"
ARCH = "zhang"
PREDICTION_TARGET = "2sbox"
TARGET_BYTE_IDX = 1
TRACE_START = 400
TRACE_END = 1500
SEED = 777

model_name = f"{IMPL}-{PREDICTION_TARGET}-byte{TARGET_BYTE_IDX}-{ARCH}-{TRACE_START}_{TRACE_END}-s{SEED}"

epoch = metadata_best_epoch(model_name)

model_path = f"models/{model_name}/epoch{epoch}.pt"
print(model_path)

model = torch.load(model_path).to()

models/fixslice-2sbox-byte1-zhang-400_1500-s777/epoch40.pt


C:\Users\Ulrik\AppData\Local\Temp\ipykernel_30488\3093016261.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(model_path).to()


In [7]:
sample = 259

traces_mean, traces_std = get_traces_mean_std(TRACE_START, TRACE_END)

traces = (val_test_traces[sample, :, TRACE_START:TRACE_END] - traces_mean) / traces_std
plaintexts_B1 = val_test_plaintexts[sample, :, :16] # first plaintext block
plaintexts_B2 = val_test_plaintexts[sample, :, 16:] # second plaintext block
key = val_test_keys[sample]

print(traces.shape)
print(plaintexts_B1.shape)
print(key.shape)

torch.Size([500, 1100])
torch.Size([500, 16])
torch.Size([16])


In [7]:
print("True key:", key.long().tolist())
true_key =  key.long().tolist()

sbox1_scores, sbox2_scores = model(traces.to(device))
numpy_scores1, numpy_scores2 = sbox1_scores.detach().cpu().numpy(),sbox2_scores.detach().cpu().numpy()

log_softmax = nn.LogSoftmax(dim=1)

for n_traces in range(2,150):
    guesses = []

    numpy_sbox1_scores_slice = numpy_scores1[:n_traces]
    numpy_sbox2_scores_slice = numpy_scores2[:n_traces]

    for subkey in range(16):

        plaintext_bytes1 = plaintexts_B1[:n_traces, subkey]
        plaintext_bytes1 = plaintext_bytes1.long().detach().cpu().numpy().squeeze()

        plaintext_bytes2 = plaintexts_B2[:n_traces, subkey]
        plaintext_bytes2 = plaintext_bytes2.long().detach().cpu().numpy().squeeze()

        numpy_keyscores1 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_bytes1, numpy_sbox1_scores_slice)
        numpy_keyscores2 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_bytes2, numpy_sbox2_scores_slice)

        x = torch.Tensor(numpy_keyscores1 + numpy_keyscores2)
        x = log_softmax(x)
        x = x.sum(dim=0)
        x = x.argmax(dim=0)

        guesses.append(x.item())

    if true_key == guesses:
        n_traces = n_traces
        print(n_traces,"traces")
        break

print("Full attack:",guesses)    

True key: [66, 59, 237, 182, 158, 214, 87, 231, 242, 124, 155, 22, 62, 118, 10, 48]
25 traces
Full attack: [66, 59, 237, 182, 158, 214, 87, 231, 242, 124, 155, 22, 62, 118, 10, 48]


In [8]:
"""Compute mean traces needed for full key recovery across N different keys, using both plaintext blocks"""

log_softmax = nn.LogSoftmax(dim=1)

traces_needed = []

for sample_idx in tqdm(range(0,500)):

    traces_ = (val_test_traces[sample_idx, :, TRACE_START:TRACE_END] - traces_mean) / traces_std
    plaintexts_B1_ = val_test_plaintexts[sample_idx, :, :16] # first plaintext block
    plaintexts_B2_ = val_test_plaintexts[sample_idx, :, 16:] # second plaintext block
    true_key_ = val_test_keys[sample_idx].long()

    sbox1_scores, sbox2_scores = model(traces_.to(device))
    numpy_scores1, numpy_scores2 = sbox1_scores.detach().cpu().numpy(),sbox2_scores.detach().cpu().numpy()

    full_guesses = torch.zeros(500,16)

    for subkey in range(16):
        plaintext_B1_bytes = plaintexts_B1_[:, subkey]
        plaintext_B1_bytes = plaintext_B1_bytes.long().detach().cpu().numpy().squeeze()

        plaintext_B2_bytes = plaintexts_B2_[:, subkey]
        plaintext_B2_bytes = plaintext_B2_bytes.long().detach().cpu().numpy().squeeze()

        numpy_keyscores1 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B1_bytes, numpy_scores1)
        numpy_keyscores2 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B2_bytes, numpy_scores2)

        numpy_keyscores3 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B1_bytes, numpy_scores2)
        numpy_keyscores4 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B2_bytes, numpy_scores1)

        x = torch.Tensor(numpy_keyscores1 + numpy_keyscores2 + numpy_keyscores3 + numpy_keyscores4)
        x = log_softmax(x)
        x = x.cumsum(dim=0)
        guesses = x.argmax(dim=1)
        full_guesses[:, subkey] = guesses

    success = (full_guesses.long() == true_key_).all(dim=1)
    n_traces = success.nonzero()[0].item() + 1
    traces_needed.append(n_traces)



mean_traces_needed = np.mean(traces_needed)

mean_traces_needed

100%|██████████| 500/500 [00:47<00:00, 10.50it/s]


20.966

In [12]:
"""Compute traces needed for 99% accurate full key recovery using single model"""

print(model_name)

log_softmax = nn.LogSoftmax(dim=1)

# Sample_idx, n_traces
success_matrix = torch.zeros(500,500)


for sample_idx in tqdm(range(0,500)):

    traces_ = (val_test_traces[sample_idx, :, TRACE_START:TRACE_END] - traces_mean) / traces_std
    plaintexts_B1_ = val_test_plaintexts[sample_idx, :, :16] # first plaintext block
    plaintexts_B2_ = val_test_plaintexts[sample_idx, :, 16:] # second plaintext block
    true_key_ = val_test_keys[sample_idx].long()

    sbox1_scores, sbox2_scores = model(traces_.to(device))
    numpy_scores1, numpy_scores2 = sbox1_scores.detach().cpu().numpy(),sbox2_scores.detach().cpu().numpy()

    full_guesses = torch.zeros(500,16)

    for subkey in range(16):
        plaintext_B1_bytes = plaintexts_B1_[:, subkey]
        plaintext_B1_bytes = plaintext_B1_bytes.long().detach().cpu().numpy().squeeze()

        plaintext_B2_bytes = plaintexts_B2_[:, subkey]
        plaintext_B2_bytes = plaintext_B2_bytes.long().detach().cpu().numpy().squeeze()

        numpy_keyscores1 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B1_bytes, numpy_scores1)
        numpy_keyscores2 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B2_bytes, numpy_scores2)

        numpy_keyscores3 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B1_bytes, numpy_scores2)
        numpy_keyscores4 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B2_bytes, numpy_scores1)

        x = torch.Tensor(numpy_keyscores1 + numpy_keyscores2 + numpy_keyscores3 + numpy_keyscores4)
        x = log_softmax(x)
        x = x.cumsum(dim=0)
        guesses = x.argmax(dim=1)
        full_guesses[:, subkey] = guesses

    
    successes = (full_guesses.long() == true_key_).all(dim=1)
    success_matrix[sample_idx, :] = successes


(success_matrix.sum(dim=0) >= 495).nonzero()[0].item()

  0%|          | 0/500 [00:00<?, ?it/s]

fixslice-2sbox-byte1-zhang-400_1500-s777


100%|██████████| 500/500 [00:48<00:00, 10.34it/s]


66

In [11]:
"""Compute traces needed for 99% accuracy on individual subkeys using single model"""


log_softmax = nn.LogSoftmax(dim=1)

# subkey, sample_idx, n_traces
success_matrix = torch.zeros(16,500,500)

for sample_idx in tqdm(range(0,500)):

    traces_ = (val_test_traces[sample_idx, :, TRACE_START:TRACE_END] - traces_mean) / traces_std
    plaintexts_B1_ = val_test_plaintexts[sample_idx, :, :16] # first plaintext block
    plaintexts_B2_ = val_test_plaintexts[sample_idx, :, 16:] # second plaintext block
    true_key_ = val_test_keys[sample_idx].long()

    sbox1_scores, sbox2_scores = model(traces_.to(device))
    numpy_scores1, numpy_scores2 = sbox1_scores.detach().cpu().numpy(),sbox2_scores.detach().cpu().numpy()

    for subkey in range(16):
        plaintext_B1_bytes = plaintexts_B1_[:, subkey]
        plaintext_B1_bytes = plaintext_B1_bytes.long().detach().cpu().numpy().squeeze()

        plaintext_B2_bytes = plaintexts_B2_[:, subkey]
        plaintext_B2_bytes = plaintext_B2_bytes.long().detach().cpu().numpy().squeeze()

        numpy_keyscores1 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B1_bytes, numpy_scores1)
        numpy_keyscores2 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B2_bytes, numpy_scores2)

        #numpy_keyscores3 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B1_bytes, numpy_scores2)
        #numpy_keyscores4 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B2_bytes, numpy_scores1)

        x = torch.Tensor(numpy_keyscores1 + numpy_keyscores2)# + numpy_keyscores3 + numpy_keyscores4)
        x = log_softmax(x)
        x = x.cumsum(dim=0)
        guesses = x.argmax(dim=1)

        success = (guesses.long() == true_key_[subkey])
        success_matrix[subkey, sample_idx] = success


n_traces_needed = []

for subkey in range(16):
    n_traces_needed.append((success_matrix[subkey].sum(dim=0) >= 495).nonzero()[0].item())

n_traces_needed

100%|██████████| 500/500 [00:35<00:00, 14.28it/s]


[31, 21, 21, 20, 30, 27, 19, 23, 24, 30, 19, 19, 50, 33, 36, 28]

In [ ]:
print("Subkey: &",  " & ".join([f"${n}$" for n in range(16)]), r"\\")
#print("\\hline")
#print("Mean traces: &", " & ".join([f"${mean:.01f}$" for mean in mean_traces_needed]), r"\\")
print("\\hline")
print("\\makecell[l]{Traces for 99\\% \\\\ accuracy} &", " & ".join([f"${n_traces:.0f}$" for n_traces in n_traces_needed]), r"\\")


#for idx, (mean, n99acc) in enumerate(zip(mean_traces_needed, traces_needed_99acc)):
#    print(f"Subkey {idx:02}, mean: {mean:.03f}, traces needed for 99%: {n99acc}")

Subkey: & $0$ & $1$ & $2$ & $3$ & $4$ & $5$ & $6$ & $7$ & $8$ & $9$ & $10$ & $11$ & $12$ & $13$ & $14$ & $15$ \\
\hline
\makecell[l]{Traces for 99\% \\ accuracy} & $30$ & $24$ & $27$ & $24$ & $53$ & $68$ & $36$ & $107$ & $51$ & $48$ & $46$ & $50$ & $98$ & $77$ & $118$ & $72$ \\
